In [33]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)

In [34]:
df = pd.read_csv("../EDA/final_model.csv")
df["Start Date"] = pd.to_datetime(df["Start Date"])
df = df.sort_values("Start Date").reset_index(drop=True)

In [35]:
df.head()

,Runs,Pos,Inns,Ground,Start Date,NotOut,Rolling5,Rolling10,Opposition_v Australia,Opposition_v Bangladesh,Opposition_v England,Opposition_v Ireland,Opposition_v Netherlands,Opposition_v New Zealand,Opposition_v Pakistan,Opposition_v South Africa,Opposition_v Sri Lanka,Opposition_v U.A.E.,Opposition_v West Indies,Opposition_v Zimbabwe
0,37,2,2,Dambulla,2008-08-20,0,12.000000,12.000000,0,0,0,0,0,0,0,0,1,0,0,0
1,25,1,1,Colombo (RPS),2008-08-24,0,24.500000,24.500000,0,0,0,0,0,0,0,0,1,0,0,0
2,54,1,1,Colombo (RPS),2008-08-27,0,24.666667,24.666667,0,0,0,0,0,0,0,0,1,0,0,0
3,31,1,2,Colombo (RPS),2008-08-29,0,32.000000,32.000000,0,0,0,0,0,0,0,0,1,0,0,0
4,2,7,1,Colombo (RPS),2009-09-14,1,31.800000,31.800000,0,0,0,0,0,0,0,0,1,0,0,0


In [36]:
opposition_cols = [
    col for col in df.columns
    if col.startswith("Opposition_")
]

In [37]:
opposition_cols

['Opposition_v Australia',
 'Opposition_v Bangladesh',
 'Opposition_v England',
 'Opposition_v Ireland',
 'Opposition_v Netherlands',
 'Opposition_v New Zealand',
 'Opposition_v Pakistan',
 'Opposition_v South Africa',
 'Opposition_v Sri Lanka',
 'Opposition_v U.A.E.',
 'Opposition_v West Indies',
 'Opposition_v Zimbabwe']

In [38]:
numeric_feats = ["Rolling5","Rolling10"]
categorical_feats = ["Ground"]

feats = numeric_feats + categorical_feats + opposition_cols

In [39]:
feats

['Rolling5',
 'Rolling10',
 'Ground',
 'Opposition_v Australia',
 'Opposition_v Bangladesh',
 'Opposition_v England',
 'Opposition_v Ireland',
 'Opposition_v Netherlands',
 'Opposition_v New Zealand',
 'Opposition_v Pakistan',
 'Opposition_v South Africa',
 'Opposition_v Sri Lanka',
 'Opposition_v U.A.E.',
 'Opposition_v West Indies',
 'Opposition_v Zimbabwe']

In [40]:
X = df[feats]
y = df["Runs"]

In [41]:
n = len(df)
n

298

In [42]:
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_end,val_end

(208, 253)

In [43]:
X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:],y.iloc[val_end:]

In [44]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_feats + opposition_cols,
    ),
    (
        "ground",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_feats,
    ),
])

In [45]:
alpha_values = [0.01,0.1,1,10,100]

In [46]:
validation_results = []

for alpha in alpha_values:
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("ridge", Ridge(alpha=alpha)),
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    validation_results.append({
        "alpha":alpha,
        "MAE": mean_absolute_error(y_val,predictions),
        "RMSE": np.sqrt(mean_squared_error(y_val,predictions)),
        "R2": r2_score(y_val, predictions),
    })


In [47]:
validation_df = pd.DataFrame(validation_results)
print(validation_df.sort_values("MAE"))

    alpha        MAE       RMSE        R2
4  100.00  33.380864  38.460536 -0.108582
3   10.00  34.179860  39.963789 -0.196935
2    1.00  35.447402  42.818013 -0.374011
1    0.10  37.288844  45.393897 -0.544301
0    0.01  37.671122  45.938176 -0.581556


In [48]:
best_alpha = validation_df.loc[validation_df["MAE"].idxmin(),"alpha"]
best_alpha

np.float64(100.0)

In [49]:
X_trainval = X.iloc[:val_end]
y_trainval =y.iloc[:val_end]

In [50]:
ridge_model = Pipeline([
    ("preprocessor",preprocessor),
    ("ridge",Ridge(alpha=best_alpha)),
])

In [51]:
ridge_model.fit(X_trainval, y_trainval)
test_predictions = ridge_model.predict(X_test)

In [52]:
test_predictions

array([45.25514679, 43.98979501, 45.90283295, 48.79111721, 47.01326289,
       54.33221283, 54.33541674, 51.07130381, 49.59230277, 51.3066899 ,
       45.33507747, 39.85916892, 40.76943326, 49.46753269, 48.72550732,
       47.084525  , 48.17194114, 43.04325596, 52.2340156 , 53.73302366,
       45.00126401, 51.14350602, 53.29758253, 27.94686945, 55.32798091,
       53.86308961, 57.37629697, 55.38005155, 53.16317727, 40.66510342,
       35.93601546, 49.87381865, 39.54696203, 49.26907643, 47.66237718,
       50.12942084, 47.30357021, 46.9283371 , 41.80224885, 47.73978953,
       48.81326407, 54.12056643, 55.07727314, 57.59494022, 55.41734964])

In [53]:
ridge_results = {
    "Model": "Ridge Regression",
    "Alpha": best_alpha,
    "MAE": mean_absolute_error(y_test, test_predictions),
    "RMSE": np.sqrt(mean_squared_error(y_test, test_predictions)),
    "R2": r2_score(y_test, test_predictions),
}

In [54]:
ridge_results

{'Model': 'Ridge Regression',
 'Alpha': np.float64(100.0),
 'MAE': 41.31299811448576,
 'RMSE': np.float64(46.88422945249324),
 'R2': -0.03516536658847769}